# CHB-MIT Seizure Detection — full pipeline

Runs the whole project end-to-end by driving the `src/` scripts as subprocesses:

1. **Encode** — `1.signal2spikes_chbmit.py` : EEG → SCTN resonator spike trains
2. **Analyze** — `2.analyze_chbmit.py` : SCTN-vs-FFT spectrograms + topography (paper Figs 4/6/7)
3. **Classify** — `3.run_classify_chbmit.py` (SVM) and `3b.run_classify_rf_chbmit.py` (Random Forest)

Scope the run with `PIPELINE_SUBJECTS` in the Config cell (default: `chb01`).
Runs locally or in Colab (see Setup). Training an SNN is out of scope here.

## Setup (Colab only — skip when running locally)

In [ ]:
# Uncomment in Google Colab. Locally, just run the Config cell below.
# !git clone https://github.com/<your-user>/Epileptic-Seizure-Classification.git
# %cd Epileptic-Seizure-Classification
# !pip -q install sctn mne numba scikit-learn scipy pandas matplotlib
# from google.colab import drive; drive.mount('/content/drive')
# # Link your CHB-MIT data (EDFs to encode, and/or pre-encoded spikes) into the repo:
# !ln -sfn /content/drive/MyDrive/Epileptic-Seizure-Classification/dataset dataset

## Config

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Repo root = parent of this notebook's src/ folder. Adjust if needed.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").is_dir() is False and (REPO_ROOT.parent / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent          # notebook opened from inside src/
os.chdir(REPO_ROOT)
PY = sys.executable

# --- scope of this run ---
os.environ["PIPELINE_SUBJECTS"]      = "chb01"        # encode these (comma-sep). "" -> all 24 (heavy!)
os.environ["PIPELINE_SUBJECT"]       = "chb01"        # recording to visualise in Stage 2
os.environ["PIPELINE_EDF"]           = "chb01_03.edf"
os.environ["PIPELINE_SEIZURE_MIN"]   = "49"
os.environ["PIPELINE_NONSEIZURE_MIN"]= "4"
CLASSIFY_PATIENTS = ["chb01"]                          # patients for Stage 3

print("Repo root :", REPO_ROOT)
print("Python    :", PY)
print("Subjects  :", os.environ["PIPELINE_SUBJECTS"])
have_edf    = (REPO_ROOT / "dataset" / "chbmit").is_dir()
have_spikes = (REPO_ROOT / "dataset" / "spikes_raw").is_dir()
print("dataset/chbmit present :", have_edf, " | dataset/spikes_raw present:", have_spikes)
assert have_edf or have_spikes, "No dataset found — mount/link dataset/ (see the Setup cell)."


def run_stage(script_args, desc, tail=6000):
    print(f"\n{'='*70}\n{desc}\n$ {PY} {' '.join(script_args)}\n{'='*70}", flush=True)
    r = subprocess.run([PY, "-u", *script_args], env=os.environ, text=True,
                       capture_output=True, errors="replace")
    out = r.stdout or ""
    print(out[-tail:])
    if r.returncode != 0:
        print("---- STDERR ----\n", (r.stderr or "")[-3000:])
        raise RuntimeError(f"{desc} failed (exit {r.returncode})")
    return out


def show_figs(*names):
    try:
        from IPython.display import Image, display
    except Exception:
        return
    for n in names:
        p = REPO_ROOT / "analysis" / "figures" / n
        if p.is_file():
            print(n); display(Image(filename=str(p)))

def show_df(df):
    try:
        from IPython.display import display; display(df)
    except Exception:
        print(df.to_string())

## Stage 1 — Encode (EEG → SCTN spikes)

Runs the encoder on `PIPELINE_SUBJECTS`. Already-encoded recordings are skipped,
so re-runs are fast. A fresh encode of one patient takes a few minutes with the
Numba fast path; encoding **all 24** patients is hours — keep the scope small here.

In [ ]:
run_stage(["src/1.signal2spikes_chbmit.py"], "STAGE 1 — ENCODE")

## Stage 2 — Analyze (SCTN vs FFT)

Reproduces the Situational-Awareness paper's analysis figures (Figs 4/6/7) for the
selected recording: FFT-vs-SCTN spectrograms, per-channel grids, and topography.
Figures are written to `analysis/figures/`.

In [ ]:
run_stage(["src/2.analyze_chbmit.py"], "STAGE 2 — ANALYZE")
show_figs("04_fft_vs_sctn_bands.png",
          "08_topography_fft_vs_sctn.png",
          "09_topography_seizure_vs_non_seizure.png")

## Stage 3 — Classify (SVM + Random Forest)

Trains the classical classifiers on the SCTN burstiness features and reports
per-patient honest (leave-one-recording-out) performance + a pooled confusion
matrix. Results go to `output_data/results/`.

In [ ]:
run_stage(["src/3.run_classify_chbmit.py",  *CLASSIFY_PATIENTS], "STAGE 3a — SVM")
run_stage(["src/3b.run_classify_rf_chbmit.py", *CLASSIFY_PATIENTS], "STAGE 3b — Random Forest")

import pandas as pd
res = REPO_ROOT / "output_data" / "results"
for name in ("svm_classifier_report.csv", "rf_classifier_report.csv"):
    p = res / name
    if p.is_file():
        print("\n" + name); show_df(pd.read_csv(p).round(3))
try:
    from IPython.display import Image, display
    for name in ("svm_confusion_matrix.png", "rf_confusion_matrix.png"):
        p = res / name
        if p.is_file():
            display(Image(filename=str(p)))
except Exception:
    pass

## Done

- Spikes  → `dataset/spikes_raw/`
- Figures → `analysis/figures/`  ·  stats → `analysis/stats/`
- Classifier reports → `output_data/results/`

To run more patients: set `PIPELINE_SUBJECTS` (and `CLASSIFY_PATIENTS`) in Config and re-run.
For the SNN classifier, see the SNN branch (out of scope for the classical-classifier pipeline).